In [1]:
import numpy as np
import MDAnalysis as mda
from MDAnalysis.transformations import unwrap, center_in_box, wrap
from MDAnalysis.lib.distances import distance_array
from MDAnalysis.analysis.leaflet import LeafletFinder

from pathlib import Path
import numpy as np

import re


In [2]:
def leaflet_resids_upper_lower(u, heads, cutoff=15.0):
    """
    Returns (upper_resids, lower_resids) based on mean z of headgroup atoms.
    """
    LF = LeafletFinder(u, heads, cutoff=cutoff)

    g0 = LF.groups(0)  # AtomGroup of head atoms in leaflet 0
    g1 = LF.groups(1)

    # Decide which is upper by mean z
    z0 = g0.positions[:, 2].mean()
    z1 = g1.positions[:, 2].mean()

    if z0 >= z1:
        upper = set(g0.resids)
        lower = set(g1.resids)
    else:
        upper = set(g1.resids)
        lower = set(g0.resids)

    return upper, lower

## Set wnt name and load the trajs

In [3]:
id_name = '1'

In [4]:
path_name = f"/gpfs/u/home/MLMS/MLMScllh/scratch/projects/WntWls_PPI/data/wnt{id_name}/copy01/"
path = Path(path_name)

top = f"Wnt{id_name}WlsPc_copy_01.psf"
top_path = path / top

# directory containing DCDs
dcd_dir = path / "dcd_files"

# find all dcd files
dcd_files = list(dcd_dir.glob("*.dcd"))

# extract trailing number for numeric sort
def dcd_index(p):
    m = re.search(r"(\d+)\.dcd$", p.name)
    if not m:
        raise ValueError(f"Cannot extract index from {p.name}")
    return int(m.group(1))

# sort from ...1.dcd to ...101.dcd
dcd_paths = sorted(dcd_files, key=dcd_index)

# drop the first 50 which may be in unstable status
dcd_paths = dcd_paths[50:]


In [ ]:
u = mda.Universe(top_path, dcd_paths)
protA = u.select_atoms("segid PROA")
protB = u.select_atoms("segid PROB")
cara = u.select_atoms("segid CARA")
complex_ab = protA + protB
others = u.atoms.difference(complex_ab + cara)


## transformation of the system, centered around the WLS

In [10]:
from MDAnalysis.transformations import TransformationBase


class CenterBoxXY(TransformationBase):
    def __init__(self):
        super().__init__()

    def _transform(self, ts):
        Lx, Ly, Lz, *_ = ts.dimensions
        ts.positions[:, 0] -= Lx / 2
        ts.positions[:, 1] -= Ly / 2
        return ts


# ----------------------------
# Define transformations
# ----------------------------

transformations = [
    unwrap(protB),  # make protein B whole
    center_in_box(protB, wrap=False),  # center protein B in the box
    wrap(
        protA, compound="segments"
    ),  # wrap mostly the protein A by segments around protein B
    wrap(
        cara, compound="segments"
    ),  # wrap the carboxyl group by segments around protein A
    #center_in_box(complex_ab, wrap=False),
    #wrap(
    #    cara, compound="segments"
    #),  # wrap the carboxyl group by segments around protein A+B
    wrap(others, compound="residues"),  # wrap everything by residue around protein
    CenterBoxXY(),  # center the system in the xy plane to (0,0)
]

u.trajectory.add_transformations(*transformations)


## define near, mid, and far regions, and separate lower/upper leaflet

In [ ]:
lipid_heads = u.select_atoms("segid MEMB and name P")  # adjust head atom name
# If you want CHOL too, pick a representative atom (e.g., ROH oxygen), or treat separately.
print(lipid_heads)
near_cut = 20.0  # Angstrom, example
far_cut = 40.0  # Angstrom, example

In [ ]:
upper_resids, lower_resids = leaflet_resids_upper_lower(u, lipid_heads)
#print(upper_resids)

upper_leaflet_sel = (
    f"segid MEMB and resid {min(upper_resids)}-{max(upper_resids)} and name P"
)
lower_leaflet_sel = (
    f"segid MEMB and resid {min(lower_resids)}-{max(lower_resids)} and name P"
)
print(upper_leaflet_sel)


upper_heads = u.select_atoms(upper_leaflet_sel)
lower_heads = u.select_atoms(lower_leaflet_sel)
print(lower_heads)

## Perform calculations

In [ ]:
import time

t0 = time.time()
t0_tmp = time.time()

apl = {
    "upper": {"near": [], "mid": [], "far": []},
    "lower": {"near": [], "mid": [], "far": []},
}

frame_ct = 0  # frame counter

for ts in u.trajectory:
#for ts in u.trajectory[::stride]:

    # --- upper leaflet assignment
    r = np.sqrt(upper_heads.positions[:, 0] ** 2 + upper_heads.positions[:, 1] ** 2)

    near = r < near_cut
    far = r > far_cut
    mid = (~near) & (~far)

    apl["upper"]["near"].append(sum(near))
    apl["upper"]["mid"].append(sum(mid))
    apl["upper"]["far"].append(sum(far))

    # --- lower leaflet assignment
    r = np.sqrt(lower_heads.positions[:, 0] ** 2 + lower_heads.positions[:, 1] ** 2)

    near = r < near_cut
    far = r > far_cut
    mid = (~near) & (~far)

    apl["lower"]["near"].append(sum(near))
    apl["lower"]["mid"].append(sum(mid))
    apl["lower"]["far"].append(sum(far))

    frame_ct += 1
    if frame_ct % 200 == 0:
        print(f"frame {frame_ct} processed")
        t_tmp = time.time()
        elapsed_tmp = (t_tmp - t0_tmp) / 60.0
        print(f"200 frames runtime: {elapsed_tmp:.2f} minutes")
        t0_tmp = time.time()

## calculate area for each region, then calculatge apl

In [ ]:
area_near = np.pi * 20 * 20
area_mid = np.pi * 40 * 40 - area_near
area_far = area - area_mid - area_near
print(area_near, area_mid, area_far)


In [16]:
apl["upper"]["near"] = np.asarray(apl["upper"]["near"], float)
apl["upper"]["mid"] = np.asarray(apl["upper"]["mid"], float)
apl["upper"]["far"] = np.asarray(apl["upper"]["far"], float)
apl["lower"]["near"] = np.asarray(apl["lower"]["near"], float)
apl["lower"]["mid"] = np.asarray(apl["lower"]["mid"], float)
apl["lower"]["far"] = np.asarray(apl["lower"]["far"], float)
# area_list = np.asarray(area_list, float)

In [ ]:
apl["upper"]["near"] = area_near / apl["upper"]["near"]
apl["upper"]["mid"] = area_mid / apl["upper"]["mid"]
apl["upper"]["far"] = area_far / apl["upper"]["far"]
apl["lower"]["near"] = area_near / apl["lower"]["near"]
apl["lower"]["mid"] = area_mid / apl["lower"]["mid"]
apl["lower"]["far"] = area_far / apl["lower"]["far"]


## Save the output

In [20]:
import numpy as np

np.save(f"apl_Wnt{id_name}.npy", apl, allow_pickle=True)

#apl = np.load(f"apl_Wnt{id_name}.npy", allow_pickle=True).item()

In [ ]:
import csv
import numpy as np

out_csv = "apl_summary_all_systems.csv"

# write header once
with open(out_csv, "a", newline="") as f:
    writer = csv.writer(f)
    # write header once
    if id_name == "1":
        writer.writerow(["system", "leaflet", "region", "mean_apl", "std_apl"])

    system_name = f"Wnt{id_name}"
    for leaflet in ["upper", "lower"]:
        for region in ["near", "mid", "far"]:
            vals = apl[leaflet][region]
            mean = float(np.mean(vals))
            std  = float(np.std(vals))
            
            writer.writerow([system_name, leaflet, region, f"{mean:.4f}", f"{std:.4f}"])

print("Saved:", out_csv)
